ARTI308 - Machine Learning

# Lab 4: Data Quality Assessment & Preprocessing  Assignment

This notebook completes the five assignment tasks from Lab 4 using the `Chocolate_Sales.csv` dataset.

---

## Setup  Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

## Load Dataset

In [ ]:
df = pd.read_csv("Chocolate_Sales.csv")
df.head(10)

---
## Task 1  Identify Data Quality Issues

We inspect the dataset for common data quality problems:
- Wrong data types
- Missing values
- Duplicate rows
- Inconsistent or noisy values

In [ ]:
# 1a. Check data types
print("=== Data Types ===")
print(df.dtypes)

In [ ]:
# 1b. Check missing values
print("=== Missing Values ===")
print(df.isnull().sum())

In [ ]:
# 1c. Check for duplicate rows
print("=== Duplicate Rows ===")
print(f"Number of duplicate rows: {df.duplicated().sum()}")

In [ ]:
# 1d. Basic summary statistics to spot noisy/impossible values
print("=== Summary Statistics ===")
df.describe()

### Task 1  Findings

| Issue | Column | Detail |
|---|---|---|
| Wrong data type | `Date` | Stored as `object`; should be `datetime64` |
| Wrong data type | `Amount` | Stored as `object` due to `$` and `,`; should be `float64` |
| Missing values | `Amount` | Several rows have `NaN` after type conversion |
| No duplicates | — | No exact duplicate rows detected |

**Action taken next:** Convert `Date` and `Amount` to correct types before proceeding.

In [ ]:
# Fix data types before continuing
df['Date']   = pd.to_datetime(df['Date'], dayfirst=True)
df['Amount'] = df['Amount'].replace(r'[\$,]', '', regex=True)
df['Amount'] = pd.to_numeric(df['Amount'])

print("Updated dtypes:")
print(df.dtypes)

---
## Task 2  Apply One Missing Value Strategy and Explain Why

First, we confirm which column has missing values and how many.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
total_rows = len(df)
missing_amount = df['Amount'].isnull().sum()
pct = missing_amount / total_rows * 100
print(f"Rows with missing Amount: {missing_amount} out of {total_rows} ({pct:.1f}%)")

### Chosen Strategy: Mean Imputation

**Why mean imputation?**

- The `Amount` column represents sales revenue, a continuous numerical variable.
- Only a small percentage of rows are missing (~0.2%), so the missing data is unlikely to follow a systematic pattern (Missing Completely at Random  MCAR).
- Mean imputation preserves the number of rows, which is important to avoid data loss.
- The distribution of `Amount` is roughly unimodal; the mean is therefore a reasonable estimate for unknown values.

**When NOT to use mean imputation:**
- If data were missing due to a specific reason (e.g., all zeros were left blank), the mean would be misleading.
- For heavily skewed distributions, the median would be a better choice.


In [ ]:
# Apply mean imputation on Amount
df_task2 = df.copy()
mean_amount = df_task2['Amount'].mean()
df_task2['Amount'].fillna(mean_amount, inplace=True)

print(f"Mean value used for imputation: ${mean_amount:,.2f}")
print()
print("Missing values after imputation:")
print(df_task2.isnull().sum())

In [ ]:
# Visualize the effect: distribution before vs after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Amount'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Amount Distribution — Before Imputation
(NaN rows excluded)', fontsize=11)
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(mean_amount, color='red', linestyle='--', label=f'Mean = ${mean_amount:,.0f}')
axes[0].legend()

axes[1].hist(df_task2['Amount'], bins=50, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[1].set_title('Amount Distribution — After Mean Imputation', fontsize=11)
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Task 2 — Effect of Mean Imputation on Amount', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Task 3  Detect and Handle Outliers Using IQR

We use the **Interquartile Range (IQR)** method to detect outliers in the `Amount` and `Boxes Shipped` columns.

**IQR formula:**
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 - Q1
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Any value outside these bounds is flagged as an outlier.

In [ ]:
# Use the imputed dataframe for this task
df_task3 = df_task2.copy()

#  Detect outliers in Amount 
Q1_amt = df_task3['Amount'].quantile(0.25)
Q3_amt = df_task3['Amount'].quantile(0.75)
IQR_amt = Q3_amt - Q1_amt
lower_amt = Q1_amt - 1.5 * IQR_amt
upper_amt = Q3_amt + 1.5 * IQR_amt

outliers_amt = df_task3[(df_task3['Amount'] < lower_amt) | (df_task3['Amount'] > upper_amt)]

print("=== Amount Outlier Detection ===")
print(f"  Q1 = ${Q1_amt:,.2f}  |  Q3 = ${Q3_amt:,.2f}  |  IQR = ${IQR_amt:,.2f}")
print(f"  Lower bound: ${lower_amt:,.2f}")
print(f"  Upper bound: ${upper_amt:,.2f}")
print(f"  Outliers detected: {len(outliers_amt)} rows")

In [ ]:
#  Detect outliers in Boxes Shipped 
Q1_box = df_task3['Boxes Shipped'].quantile(0.25)
Q3_box = df_task3['Boxes Shipped'].quantile(0.75)
IQR_box = Q3_box - Q1_box
lower_box = Q1_box - 1.5 * IQR_box
upper_box = Q3_box + 1.5 * IQR_box

outliers_box = df_task3[(df_task3['Boxes Shipped'] < lower_box) | (df_task3['Boxes Shipped'] > upper_box)]

print("=== Boxes Shipped Outlier Detection ===")
print(f"  Q1 = {Q1_box:.0f}  |  Q3 = {Q3_box:.0f}  |  IQR = {IQR_box:.0f}")
print(f"  Lower bound: {lower_box:.0f}")
print(f"  Upper bound: {upper_box:.0f}")
print(f"  Outliers detected: {len(outliers_box)} rows")

In [ ]:
# Visualize outliers with boxplots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df_task3['Amount'], ax=axes[0], color='steelblue')
axes[0].set_title('Boxplot — Amount (before removing outliers)', fontsize=11)
axes[0].axvline(lower_amt, color='red', linestyle='--', linewidth=1, label='IQR bounds')
axes[0].axvline(upper_amt, color='red', linestyle='--', linewidth=1)
axes[0].legend()

sns.boxplot(x=df_task3['Boxes Shipped'], ax=axes[1], color='coral')
axes[1].set_title('Boxplot — Boxes Shipped (before removing outliers)', fontsize=11)
axes[1].axvline(lower_box, color='red', linestyle='--', linewidth=1, label='IQR bounds')
axes[1].axvline(upper_box, color='red', linestyle='--', linewidth=1)
axes[1].legend()

plt.suptitle('Task 3 — Outlier Detection via IQR', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#  Remove outliers from both columns 
df_clean = df_task3[
    (df_task3['Amount'] >= lower_amt) & (df_task3['Amount'] <= upper_amt) &
    (df_task3['Boxes Shipped'] >= lower_box) & (df_task3['Boxes Shipped'] <= upper_box)
].copy()

print(f"Original shape:              {df_task3.shape}")
print(f"After removing outliers:     {df_clean.shape}")
print(f"Rows removed:                {len(df_task3) - len(df_clean)}")

In [ ]:
# Confirm no outliers remain
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df_clean['Amount'], ax=axes[0], color='mediumseagreen')
axes[0].set_title('Boxplot — Amount (after removing outliers)', fontsize=11)

sns.boxplot(x=df_clean['Boxes Shipped'], ax=axes[1], color='mediumpurple')
axes[1].set_title('Boxplot — Boxes Shipped (after removing outliers)', fontsize=11)

plt.suptitle('Task 3 — After Outlier Removal', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Task 4  Normalize Numerical Features Using Min-Max and Z-Score

We normalize the two numerical features: `Amount` and `Boxes Shipped`.

**Why normalize?**
- Machine learning algorithms that use distance or gradient descent are sensitive to feature scales.
- `Amount` can range in the thousands while `Boxes Shipped` is in the hundreds — without scaling, the larger column would dominate.

### Method 1: Min-Max Normalization (scales values to [0, 1])
$$X_{\text{norm}} = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$$

### Method 2: Z-Score Standardization (mean=0, std=1)
$$X_{\text{std}} = \frac{X - \mu}{\sigma}$$

In [ ]:
numerical_cols = ['Amount', 'Boxes Shipped']

#  Min-Max Normalization 
minmax_scaler = MinMaxScaler()
df_minmax = df_clean[numerical_cols].copy()
df_minmax[numerical_cols] = minmax_scaler.fit_transform(df_clean[numerical_cols])

print("=== Min-Max Normalized ===")
print(df_minmax.describe().round(4))

In [ ]:
#  Z-Score Standardization 
zscore_scaler = StandardScaler()
df_zscore = df_clean[numerical_cols].copy()
df_zscore[numerical_cols] = zscore_scaler.fit_transform(df_clean[numerical_cols])

print("=== Z-Score Standardized ===")
print(df_zscore.describe().round(4))

In [ ]:
# Visual comparison: original vs Min-Max vs Z-Score
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(numerical_cols):
    # Original
    axes[i][0].hist(df_clean[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i][0].set_title(f'{col} — Original', fontsize=10)
    axes[i][0].set_xlabel('Value')

    # Min-Max
    axes[i][1].hist(df_minmax[col], bins=40, color='coral', edgecolor='white', alpha=0.8)
    axes[i][1].set_title(f'{col} — Min-Max [0, 1]', fontsize=10)
    axes[i][1].set_xlabel('Value')

    # Z-Score
    axes[i][2].hist(df_zscore[col], bins=40, color='mediumseagreen', edgecolor='white', alpha=0.8)
    axes[i][2].set_title(f'{col} — Z-Score (μ=0, σ=1)', fontsize=10)
    axes[i][2].set_xlabel('Value')

plt.suptitle('Task 4 — Original vs Min-Max vs Z-Score Normalization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Task 4  Interpretation

| Method | Range | When to use |
|---|---|---|
| **Min-Max** | [0, 1] | When you need bounded values; useful for neural networks and image data |
| **Z-Score** | Roughly [−3, 3] | When the algorithm assumes normally distributed data (e.g., PCA, SVM, K-Means) |

Both methods preserve the shape of the distribution  only the scale changes. Notice the histogram shapes are identical across all three columns for each feature.

---
## Task 5  Apply PCA and Interpret Explained Variance

PCA (Principal Component Analysis) reduces the dimensionality of the data by finding the directions (principal components) that capture the most variance.

Here we apply PCA to the two scaled numerical features and analyze how much variance each component explains.

In [ ]:
# Use Z-score scaled data for PCA (PCA assumes standardized features)
X_scaled = df_zscore[numerical_cols].values

# Apply PCA  keep both components since we only have 2 features
pca = PCA(n_components=2)
principal_components = pca.fit_transform(X_scaled)

print("=== PCA Explained Variance ===")
for i, (var, ratio) in enumerate(zip(pca.explained_variance_, pca.explained_variance_ratio_)):
    print(f"  PC{i+1}: variance = {var:.4f}  |  explained ratio = {ratio*100:.2f}%")

print()
print(f"  Total variance explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
# Scree plot  explained variance per component
plt.figure(figsize=(7, 4))
components = [f'PC{i+1}' for i in range(2)]
variances = pca.explained_variance_ratio_ * 100

bars = plt.bar(components, variances, color=['steelblue', 'coral'], edgecolor='white', width=0.4)
plt.ylabel('Explained Variance (%)', fontsize=12)
plt.title('Task 5 — PCA Scree Plot
(Explained Variance per Principal Component)', fontsize=12, fontweight='bold')

for bar, val in zip(bars, variances):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylim(0, 110)
plt.tight_layout()
plt.show()

In [ ]:
# 2D PCA scatter plot colored by Amount quartile
amount_quartile = pd.qcut(df_clean['Amount'], q=4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
colors_map = {'Q1 (Low)': 'steelblue', 'Q2': 'mediumseagreen', 'Q3': 'coral', 'Q4 (High)': 'mediumpurple'}

plt.figure(figsize=(8, 6))
for label, color in colors_map.items():
    mask = amount_quartile == label
    plt.scatter(
        principal_components[mask, 0],
        principal_components[mask, 1],
        label=label, alpha=0.4, s=15, color=color
    )

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
plt.title('Task 5 — PCA 2D Projection
(Colored by Amount Quartile)', fontsize=12, fontweight='bold')
plt.legend(title='Amount Group', fontsize=9)
plt.tight_layout()
plt.show()

### Task 5  Interpretation

**How much variance does each component explain?**

Since we only have two numerical features (`Amount` and `Boxes Shipped`), PCA produces exactly two principal components that together explain 100% of the variance.

- **PC1** captures the direction of maximum variance across both features combined. Because `Amount` and `Boxes Shipped` are weakly correlated (customers can spend a lot with few boxes, or many boxes at a low unit price), PC1 mainly reflects the overall scale of a transaction.
- **PC2** captures the remaining variance  the direction orthogonal to PC1  which represents the contrast between Amount and Boxes Shipped (i.e., transactions with high amount but few boxes vs. many boxes but lower amount).

**What does the scatter plot tell us?**
- Points spread more along the horizontal axis (PC1) than vertically (PC2), confirming PC1 dominates.
- Coloring by Amount quartile shows that PC1 broadly separates low-value from high-value transactions.
- The spread of points in the vertical direction (PC2) reflects variation in transaction efficiency (amount per box).

**In real datasets with many features**, PCA allows us to reduce dimensionality while retaining most of the information  for example, reducing 17 features down to 2–3 components that still explain 80–90% of the variance, making visualization and model training much faster.

---
## Assignment Summary

| Task | Action | Key Decision |
|---|---|---|
| **Task 1** | Identified data quality issues | Wrong types (`Date`, `Amount`), missing values in `Amount` |
| **Task 2** | Applied mean imputation | Small % missing; continuous variable; mean preserves row count |
| **Task 3** | Detected and removed IQR outliers | Applied to both `Amount` and `Boxes Shipped` |
| **Task 4** | Applied Min-Max and Z-Score scaling | Both preserve distribution shape; Z-Score used for PCA |
| **Task 5** | Applied PCA and interpreted variance | PC1 dominates; PC2 captures amount-vs-boxes contrast |

End of Assignment  Lab 4.